# Santander — All-In-One Final Notebook
## Everything in one place:
- Loads existing OOF pickles (xgb, lgbm, cat, rf, nn)
- Optionally retrains any individual model with better params
- Builds augmented features from scratch
- Blends everything with Nelder-Mead optimal weights
- Applies post-processing rules
- Saves final submission

## To improve an individual model:
Set the corresponding RETRAIN flag to True and update its params.
The new OOF replaces the pickle version automatically.

In [18]:
# ============================================================
# CONFIGURATION — edit this section only
# ============================================================

PICKLE_PATH = '/kaggle/input/datasets/salonij27/finalpickles/'   # your dataset path

# Set True to retrain that model fresh instead of using pickle
RETRAIN_XGB  = False
RETRAIN_LGBM = False
RETRAIN_CAT  = False
RETRAIN_RF   = False
RETRAIN_NN   = False

# Always trains fresh (it's our new model, no pickle exists)
TRAIN_LGBM_AUG = True

N_FOLDS = 5
RANDOM_STATE = 42

In [19]:
import numpy as np
import pandas as pd
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print('All imports OK')

All imports OK


In [20]:
# ============================================================
# LOAD DATA
# ============================================================
train = pd.read_pickle(PICKLE_PATH + 'train_clean.pkl')
test  = pd.read_pickle(PICKLE_PATH + 'test_clean.pkl')
y     = train['TARGET']
train_ids = train['ID'].values
test_ids  = test['ID'].values

feat_cols = [c for c in train.columns if c not in ['ID', 'TARGET']]
X_base      = train[feat_cols].values
X_test_base = test[feat_cols].values

print(f'train: {train.shape} | test: {test.shape}')
print(f'Base features: {len(feat_cols)}')
print(f'Class balance: {y.value_counts(normalize=True).to_dict()}')

train: (76020, 97) | test: (75818, 96)
Base features: 95
Class balance: {0: 0.9604314654038411, 1: 0.0395685345961589}


In [21]:
# ============================================================
# BUILD AUGMENTED FEATURES (always done, used by LGBM_AUG)
# ============================================================
def augment(df):
    df = df.copy()
    fc = [c for c in df.columns if c not in ['ID', 'TARGET']]
    X  = df[fc]

    # Row statistics
    df['count_zeros']    = (X == 0).sum(axis=1)
    df['count_zeros_sq'] = df['count_zeros'] ** 2
    df['num_nonzero']    = (X != 0).sum(axis=1)
    df['row_std']        = X.std(axis=1)
    df['row_max']        = X.max(axis=1)

    # Age rules
    if 'var15' in df.columns:
        df['is_young']   = (df['var15'] < 23).astype(int)
        df['is_elderly'] = (df['var15'] > 80).astype(int)
        df['var15_bin']  = pd.cut(df['var15'],
                                  bins=[0,23,40,60,80,999],
                                  labels=[0,1,2,3,4]).astype(float)

    # var38 cleaning + log
    if 'var38' in df.columns:
        v38 = df['var38'].replace(117310.979016494, np.nan)
        df['var38_log']     = np.log1p(v38.fillna(v38.median()).clip(lower=0))
        df['var38_is_mode'] = (df['var38'] == 117310.979016494).astype(int)

    # Saldo log transforms
    saldo_cols = [c for c in fc if 'saldo_' in c]
    for col in saldo_cols:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))
    if saldo_cols:
        df['saldo_total']     = df[saldo_cols].sum(axis=1)
        df['saldo_total_log'] = np.log1p(df['saldo_total'].clip(lower=0))
    if 'saldo_var30' in df.columns:
        df['saldo_var30_zero'] = (df['saldo_var30'] == 0).astype(int)

    # Temporal deltas
    for base in ['num_var22', 'saldo_medio_var5', 'num_var45']:
        c1, c3 = f'{base}_ult1', f'{base}_ult3'
        if c1 in df.columns and c3 in df.columns:
            df[f'{base}_delta'] = df[c1] - df[c3]
            df[f'{base}_ratio'] = df[c1] / (df[c3].replace(0, np.nan)).fillna(1)

    return df

train_aug = augment(train)
test_aug  = augment(test)

aug_cols      = [c for c in train_aug.columns if c not in ['ID','TARGET']]
X_aug         = train_aug[aug_cols].values
X_test_aug    = test_aug[[c for c in aug_cols if c in test_aug.columns]].values

print(f'Augmented features: {len(aug_cols)} (base was {len(feat_cols)})')

Augmented features: 128 (base was 95)


In [22]:
# ============================================================
# HELPER: run CV for any model
# ============================================================
def run_cv(model_fn, X, X_test, y, name, scale=False):
    """
    model_fn: callable that returns a fresh model instance
    scale: True for MLP/LogReg (needs StandardScaler)
    Returns: oof_preds (len=train), test_preds (len=test)
    """
    oof  = np.zeros(len(y))
    test_preds = np.zeros(X_test.shape[0])
    fold_aucs = []

    for fold_i, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y.values[tr_idx], y.values[val_idx]
        X_te = X_test.copy()

        if scale:
            sc = StandardScaler()
            X_tr  = sc.fit_transform(X_tr)
            X_val = sc.transform(X_val)
            X_te  = sc.transform(X_te)

        model = model_fn()

        # XGBoost early stopping
        if isinstance(model, XGBClassifier):
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        # LightGBM early stopping
        elif isinstance(model, LGBMClassifier):
            model.fit(X_tr, y_tr,
                      eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                 lgb.log_evaluation(500)])
        # CatBoost early stopping
        elif isinstance(model, CatBoostClassifier):
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        else:
            model.fit(X_tr, y_tr)

        oof[val_idx]  = model.predict_proba(X_val)[:, 1]
        test_preds   += model.predict_proba(X_te)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y_val, oof[val_idx])
        fold_aucs.append(fold_auc)
        print(f'  [{name}] Fold {fold_i+1}: {fold_auc:.5f}')

    cv_auc = roc_auc_score(y, oof)
    print(f'  [{name}] CV AUC: {cv_auc:.5f} ± {np.std(fold_aucs):.5f}\n')
    return oof, test_preds, cv_auc

In [23]:
# ============================================================
# MODEL PARAMS — edit these to improve individual models
# ============================================================

# ── XGBoost ──
# To improve: increase n_estimators, tune max_depth/subsample
def make_xgb():
    return XGBClassifier(
        n_estimators        = 3000,
        max_depth           = 5,
        learning_rate       = 0.0162,
        subsample           = 0.70,
        colsample_bytree    = 0.70,
        scale_pos_weight    = 24,
        gamma               = 0.1,
        min_child_weight    = 5,
        reg_alpha           = 0.1,
        reg_lambda          = 1.0,
        objective           = 'binary:logistic',
        eval_metric         = 'auc',
        early_stopping_rounds = 50,
        tree_method         = 'hist',
        random_state        = RANDOM_STATE,
        n_jobs              = -1,
        verbosity           = 0,
    )

# ── LightGBM (Shiv's best params) ──
# To improve: try num_leaves 200-600, min_child_samples
def make_lgbm():
    return LGBMClassifier(
        objective           = 'binary',
        metric              = 'auc',
        num_leaves          = 466,
        learning_rate       = 0.005,
        n_estimators        = 5000,
        feature_fraction    = 0.80,
        bagging_fraction    = 0.80,
        bagging_freq        = 5,
        min_child_samples   = 20,
        reg_alpha           = 0.1,
        reg_lambda          = 1.0,
        is_unbalance        = True,
        n_jobs              = -1,
        random_state        = RANDOM_STATE,
        verbose             = -1,
    )

# ── CatBoost ──
# To improve: try depth 7-9, l2_leaf_reg, iterations
def make_cat():
    return CatBoostClassifier(
        iterations          = 3000,
        depth               = 6,
        learning_rate       = 0.05,
        l2_leaf_reg         = 3,
        eval_metric         = 'AUC',
        early_stopping_rounds = 100,
        auto_class_weights  = 'Balanced',
        random_seed         = RANDOM_STATE,
        verbose             = False,
    )

# ── Random Forest ──
# To improve: n_estimators 500-1000, max_depth, min_samples_leaf
def make_rf():
    return RandomForestClassifier(
        n_estimators        = 500,
        max_depth           = 15,
        max_features        = 'sqrt',
        min_samples_leaf    = 5,
        min_samples_split   = 10,
        class_weight        = 'balanced',
        n_jobs              = -1,
        random_state        = RANDOM_STATE,
    )

# ── MLP Neural Network ──
# To improve: hidden_layer_sizes, alpha, learning_rate_init
def make_nn():
    return MLPClassifier(
        hidden_layer_sizes  = (256, 128, 64),
        activation          = 'relu',
        solver              = 'adam',
        alpha               = 0.001,
        learning_rate_init  = 0.001,
        max_iter            = 200,
        early_stopping      = True,
        random_state        = RANDOM_STATE,
    )

# ── LGBM on augmented features (our new model) ──
def make_lgbm_aug():
    return LGBMClassifier(
        objective           = 'binary',
        metric              = 'auc',
        num_leaves          = 466,
        learning_rate       = 0.005,
        n_estimators        = 5000,
        feature_fraction    = 0.80,
        bagging_fraction    = 0.80,
        bagging_freq        = 5,
        min_child_samples   = 20,
        reg_alpha           = 0.1,
        reg_lambda          = 1.0,
        is_unbalance        = True,
        n_jobs              = -1,
        random_state        = RANDOM_STATE,
        verbose             = -1,
    )

print('Model definitions ready')

Model definitions ready


In [24]:
# ============================================================
# LOAD OR RETRAIN EACH MODEL
# ============================================================
oofs  = {}
tests = {}
aucs  = {}

# Column names inside each pickle
pickle_col = {
    'xgb':  ('xgb_oof.pkl',       'xgb_pred',  'xgb_test_preds.pkl'),
    'lgbm': ('lgbm_oof.pkl',      'lgbm_pred', 'lgbm_test_preds.pkl'),
    'cat':  ('catboost_oof.pkl',  'cat_pred',  'catboost_test_preds.pkl'),
    'rf':   ('rf_oof.pkl',        'rf_pred',   'rf_test_preds.pkl'),
    'nn':   ('nn_oof.pkl',        'nn_pred',   'nn_test_preds.pkl'),
}

retrain_flags = {
    'xgb':  (RETRAIN_XGB,  make_xgb,  X_base,     X_test_base, False),
    'lgbm': (RETRAIN_LGBM, make_lgbm, X_base,     X_test_base, False),
    'cat':  (RETRAIN_CAT,  make_cat,  X_base,     X_test_base, False),
    'rf':   (RETRAIN_RF,   make_rf,   X_base,     X_test_base, False),
    'nn':   (RETRAIN_NN,   make_nn,   X_base,     X_test_base, True),
}

print('=== LOADING / TRAINING BASE MODELS ===')
for name, (retrain, model_fn, X, X_te, scale) in retrain_flags.items():
    if retrain:
        print(f'  {name}: RETRAINING...')
        oof, test_p, auc = run_cv(model_fn, X, X_te, y, name, scale=scale)
    else:
        oof_f, col, test_f = pickle_col[name]
        try:
            oof_df   = pd.read_pickle(PICKLE_PATH + oof_f)
            test_df  = pd.read_pickle(PICKLE_PATH + test_f)
            oof      = oof_df.set_index('ID').reindex(train_ids)[col].values
            test_p   = test_df.set_index('ID').reindex(test_ids)[col].values
            auc      = roc_auc_score(y, oof)
            print(f'  {name}: loaded from pickle | CV AUC = {auc:.5f}')
        except Exception as e:
            print(f'  {name}: pickle failed ({e}) — retraining...')
            oof, test_p, auc = run_cv(model_fn, X, X_te, y, name, scale=scale)

    oofs[name]  = oof
    tests[name] = test_p
    aucs[name]  = auc

=== LOADING / TRAINING BASE MODELS ===
  xgb: loaded from pickle | CV AUC = 0.83469
  lgbm: loaded from pickle | CV AUC = 0.84101
  cat: loaded from pickle | CV AUC = 0.83853
  rf: loaded from pickle | CV AUC = 0.83851
  nn: loaded from pickle | CV AUC = 0.82589


In [25]:
# ── Train LGBM on augmented features ──
if TRAIN_LGBM_AUG:
    print('=== TRAINING LGBM_AUG (augmented features) ===')
    oof_aug, test_aug_p, auc_aug = run_cv(
        make_lgbm_aug, X_aug, X_test_aug, y, 'lgbm_aug', scale=False
    )
    oofs['lgbm_aug']  = oof_aug
    tests['lgbm_aug'] = test_aug_p
    aucs['lgbm_aug']  = auc_aug

=== TRAINING LGBM_AUG (augmented features) ===
  [lgbm_aug] Fold 1: 0.81887
  [lgbm_aug] Fold 2: 0.83439
  [lgbm_aug] Fold 3: 0.82854
  [lgbm_aug] Fold 4: 0.81742
  [lgbm_aug] Fold 5: 0.82935
  [lgbm_aug] CV AUC: 0.80803 ± 0.00651



In [26]:
# ============================================================
# SUMMARY OF ALL MODELS
# ============================================================
print('=== ALL MODEL CV AUCs ===')
for name, auc in sorted(aucs.items(), key=lambda x: -x[1]):
    source = 'retrained' if (
        (name == 'lgbm_aug') or
        retrain_flags.get(name, (False,))[0]
    ) else 'pickle'
    print(f'  {name:12s}: {auc:.5f}  [{source}]')

# Check correlations — high corr = redundant models
names      = list(oofs.keys())
oof_matrix = np.column_stack([oofs[n] for n in names])
test_matrix= np.column_stack([tests[n] for n in names])

print('\nModel OOF correlations:')
print(pd.DataFrame(oof_matrix, columns=names).corr().round(3))

=== ALL MODEL CV AUCs ===
  lgbm        : 0.84101  [pickle]
  cat         : 0.83853  [pickle]
  rf          : 0.83851  [pickle]
  xgb         : 0.83469  [pickle]
  nn          : 0.82589  [pickle]
  lgbm_aug    : 0.80803  [retrained]

Model OOF correlations:
            xgb   lgbm    cat     rf     nn  lgbm_aug
xgb       1.000  0.974  0.960  0.946  0.896     0.750
lgbm      0.974  1.000  0.973  0.975  0.915     0.772
cat       0.960  0.973  1.000  0.953  0.910     0.749
rf        0.946  0.975  0.953  1.000  0.915     0.785
nn        0.896  0.915  0.910  0.915  1.000     0.748
lgbm_aug  0.750  0.772  0.749  0.785  0.748     1.000


In [27]:
# ============================================================
# NELDER-MEAD OPTIMAL BLEND
# ============================================================
def neg_auc(w):
    sw = np.exp(w) / np.exp(w).sum()
    return -roc_auc_score(y, oof_matrix @ sw)

res = minimize(neg_auc, np.zeros(len(names)), method='Nelder-Mead',
               options={'maxiter': 50000, 'xatol': 1e-10, 'fatol': 1e-10})

w_opt      = np.exp(res.x) / np.exp(res.x).sum()
blend_oof  = oof_matrix  @ w_opt
blend_test = test_matrix @ w_opt
blend_auc  = roc_auc_score(y, blend_oof)

print(f'Ensemble CV AUC: {blend_auc:.7f}')
print('Optimal weights:')
for n, w in sorted(zip(names, w_opt), key=lambda x: -x[1]):
    print(f'  {n:12s}: {w:.4f}')

Ensemble CV AUC: 0.8414194
Optimal weights:
  cat         : 0.2594
  nn          : 0.2145
  lgbm        : 0.1791
  rf          : 0.1677
  xgb         : 0.1499
  lgbm_aug    : 0.0294


In [28]:
# ============================================================
# POST-PROCESSING + SAVE SUBMISSION
# ============================================================
def post_process(preds, df):
    p = preds.copy()
    # Rule 1: no unsatisfied customers under age 23 (EDA finding)
    if 'var15' in df.columns:
        p[df['var15'].values < 23] = 0.0
    # Rule 2: very high cash balance = clearly satisfied
    if 'saldo_var30' in df.columns:
        p[df['saldo_var30'].values > 500000] = 0.0
    return p

final_preds = post_process(blend_test, test)

# Validate
assert len(final_preds) == len(test_ids),  'Shape mismatch!'
assert not np.isnan(final_preds).any(),    'NaN in predictions!'
assert final_preds.min() >= 0,             'Negative predictions!'
assert final_preds.max() <= 1,             'Predictions > 1!'

sub = pd.DataFrame({'ID': test_ids, 'TARGET': final_preds})
sub.to_csv('submission_allinone.csv', index=False)

print(f'Saved submission_allinone.csv')
print(f'Shape: {sub.shape}')
print(f'Mean prediction: {final_preds.mean():.4f} (expect ~0.04)')
print(f'Ensemble CV AUC: {blend_auc:.5f}')
print(f'Your current best private LB: 0.82745')
print(f'Submit this and compare!')

Saved submission_allinone.csv
Shape: (75818, 2)
Mean prediction: 0.0403 (expect ~0.04)
Ensemble CV AUC: 0.84142
Your current best private LB: 0.82745
Submit this and compare!
